In [4]:
import os
import csv
import json
import shutil
import zipfile
import subprocess
import tempfile
from pathlib import Path
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed


In [5]:
from pathlib import Path

# INPUT
SOURCE = Path.home() / "ocr_RAG" / "ncst_files"

# OUTPUT
OUTPUT = Path.home() / "ocr_RAG"/ "ncst_processed"

SOURCE = SOURCE.resolve()
OUTPUT = OUTPUT.resolve()

OUTPUT.mkdir(parents=True, exist_ok=True)

print("Source :", SOURCE)
print("Output :", OUTPUT)

Source : /home/s4lm-xi/ocr_RAG/ncst_files
Output : /home/s4lm-xi/ocr_RAG/ncst_processed


In [6]:

from tqdm.auto import tqdm


# =========================================================
# CONFIG
# =========================================================

# 11 CPU threads available → leave 2 unused
MAX_WORKERS = max(1, (os.cpu_count() or 11) - 2)

# ---------------------------------------------------------
# Supported formats
# ---------------------------------------------------------

OFFICE_TO_PDF = {
    ".doc": "Word",
    ".docx": "Word",
    ".ppt": "PowerPoint",
    ".pptx": "PowerPoint",
}

KEEP_AS_PDF = {
    ".pdf",
}

# Files we intentionally do NOT process
IGNORE_EXTENSIONS = {
    ".png",
    ".jpg",
    ".jpeg",
    ".gif",
    ".bmp",
    ".webp",
    ".tif",
    ".tiff",
    ".cr2",
    ".ds_store",
    ".vsdx",
    ".xlsx",
    ".csv",
}

IGNORE_FILENAMES = {
    ".ds_store",
}

In [7]:
# =========================================================
# TEMP FILE CHECK
# =========================================================

def is_temp_file(path):
    return (
        path.name.startswith("~$")
        or path.name.startswith(".~lock.")
        or path.name in IGNORE_FILENAMES
    )


# =========================================================
# LIBREOFFICE
# =========================================================

LIBREOFFICE = (
    shutil.which("libreoffice")
    or shutil.which("soffice")
)

if not LIBREOFFICE:
    raise RuntimeError(
        "LibreOffice not found. Run:\n"
        "sudo apt update && sudo apt install -y libreoffice"
    )

print(f"LibreOffice: {LIBREOFFICE}")
print(f"CPU threads : {os.cpu_count()}")
print(f"Workers     : {MAX_WORKERS}")
print()

LibreOffice: /usr/bin/libreoffice
CPU threads : 12
Workers     : 10



In [8]:






# =========================================================
# LIBREOFFICE CONVERSION
# =========================================================

def convert_to_pdf(input_file: Path, output_dir: Path):
    """
    Convert Office file to PDF using LibreOffice.
    Returns generated PDF path.
    """

    output_dir.mkdir(parents=True, exist_ok=True)

    cmd = [
        LIBREOFFICE,
        "--headless",
        "--convert-to",
        "pdf",
        "--outdir",
        str(output_dir),
        str(input_file),
    ]

    result = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        timeout=300,
    )

    expected = output_dir / f"{input_file.stem}.pdf"

    if expected.exists():
        return expected

    candidates = list(output_dir.glob("*.pdf"))

    if candidates:
        candidates.sort(
            key=lambda p: p.stat().st_mtime,
            reverse=True,
        )
        return candidates[0]

    raise RuntimeError(
        "LibreOffice failed\n"
        f"stdout: {result.stdout}\n"
        f"stderr: {result.stderr}"
    )


# =========================================================
# ZIP EXTRACTION
# =========================================================

def safe_extract_zip(zip_path: Path, destination: Path):
    """
    Safely extract ZIP files without allowing path traversal.
    """

    destination.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as z:
        destination_resolved = destination.resolve()

        for member in z.infolist():

            target = (
                destination / member.filename
            ).resolve()

            if not str(target).startswith(
                str(destination_resolved)
            ):
                raise RuntimeError(
                    f"Unsafe ZIP path detected: {member.filename}"
                )

            z.extract(member, destination)


# =========================================================
# LOGGING
# =========================================================

results = []


def log_result(
    source,
    output,
    status,
    file_type,
    error="",
):
    return {
        "source": str(source),
        "output": str(output) if output else "",
        "status": status,
        "type": file_type,
        "error": error,
    }


# =========================================================
# PROCESS ONE FILE
# =========================================================

def process_file(source_file: Path, relative_root: Path):

    rel = source_file.relative_to(relative_root)

    # -----------------------------------------------------
    # Temporary Office files
    # -----------------------------------------------------

    if is_temp_file(source_file):
        return log_result(
            source_file,
            None,
            "skipped",
            "temporary",
            "Temporary/lock file",
        )

    suffix = source_file.suffix.lower()

    # -----------------------------------------------------
    # Existing PDF
    # -----------------------------------------------------

    if suffix in KEEP_AS_PDF:

        destination = OUTPUT / rel

        destination.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        if destination.exists():
            return log_result(
                source_file,
                destination,
                "already_exists",
                "PDF",
            )

        shutil.copy2(
            source_file,
            destination,
        )

        return log_result(
            source_file,
            destination,
            "copied",
            "PDF",
        )

    # -----------------------------------------------------
    # Ignored files
    # -----------------------------------------------------

    if suffix in IGNORE_EXTENSIONS:

        return log_result(
            source_file,
            None,
            "skipped",
            suffix or "no_extension",
            "Ignored file type",
        )

    # -----------------------------------------------------
    # Office → PDF
    # -----------------------------------------------------

    if suffix in OFFICE_TO_PDF:

        relative_pdf = rel.with_suffix(".pdf")

        destination = (
            OUTPUT / relative_pdf
        )

        destination.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        # Already processed
        if destination.exists():

            return log_result(
                source_file,
                destination,
                "already_exists",
                OFFICE_TO_PDF[suffix],
            )

        # Each thread gets its own temporary directory.
        # This prevents LibreOffice workers from colliding.
        with tempfile.TemporaryDirectory() as tmp:

            tmp_dir = Path(tmp)

            try:

                generated = convert_to_pdf(
                    source_file,
                    tmp_dir,
                )

                shutil.move(
                    str(generated),
                    str(destination),
                )

                return log_result(
                    source_file,
                    destination,
                    "converted",
                    OFFICE_TO_PDF[suffix],
                )

            except Exception as e:

                return log_result(
                    source_file,
                    None,
                    "FAILED",
                    OFFICE_TO_PDF[suffix],
                    str(e),
                )

    # -----------------------------------------------------
    # Unknown
    # -----------------------------------------------------

    return log_result(
        source_file,
        None,
        "skipped",
        suffix or "no_extension",
        "Unsupported file type",
    )


# =========================================================
# MULTI-THREADED RECURSIVE SCANNER
# =========================================================

def process_tree(source_root: Path):

    files = [
        p
        for p in source_root.rglob("*")
        if p.is_file()
    ]

    print(
        f"Found {len(files):,} files"
    )

    print(
        f"Using {MAX_WORKERS} parallel workers "
        f"(leaving ~2 CPU threads free)"
    )

    print()
    print("Processing...")
    print()

    completed = 0

    with ThreadPoolExecutor(
        max_workers=MAX_WORKERS
    ) as executor:

        futures = {
            executor.submit(
                process_file,
                file_path,
                source_root,
            ): file_path
            for file_path in files
        }

        with tqdm(
            total=len(futures),
            desc="Processing",
            unit="file",
            dynamic_ncols=True,
        ) as progress:

            for future in as_completed(futures):

                file_path = futures[future]

                try:

                    result = future.result()

                except Exception as e:

                    result = log_result(
                        file_path,
                        None,
                        "FAILED",
                        file_path.suffix.lower(),
                        str(e),
                    )

                results.append(result)

                completed += 1
                progress.update(1)

    print()
    print("Processing complete.")
    print(
        f"Processed {completed:,} files."
    )


# =========================================================
# RUN
# =========================================================

print("Starting preprocessing...")
print()

process_tree(SOURCE)

print()
print("Done.")

Starting preprocessing...

Found 918 files
Using 10 parallel workers (leaving ~2 CPU threads free)

Processing...



Processing:   0%|                                                                                 | 0/918 [00:…


Processing complete.
Processed 918 files.

Done.
